# UAV Neo — Async Core Test

This notebook tests each sensor module and the Coral EdgeTPU inference using the async API.
Run each cell in order. The drone must be powered on with the teleop stack running:

```bash
ros2 launch uav_neo_ros2_driver teleop.launch.py edgetpu_enable:=true
```

## 1. Initialize Drone

In [1]:
import sys
sys.path.insert(0, "../../library")
sys.path.insert(0, "../../library/real")

import drone_core
import drone_utils as uav_utils
import numpy as np
import cv2 as cv
from IPython.display import display, HTML
import ipywidgets as widgets
from PIL import Image as PILImage
import io, time

In [2]:
def to_jpeg_bytes(bgr_image, resize=(320, 240)):
    """Convert a BGR numpy image to JPEG bytes for widget display."""
    if resize:
        bgr_image = cv.resize(bgr_image, resize)
    _, buf = cv.imencode('.jpg', bgr_image, [cv.IMWRITE_JPEG_QUALITY, 70])
    return buf.tobytes()

In [ ]:
# Create the drone in real mode and start the async ROS2 executor
drone = drone_core.create_drone(isSimulation=False)
drone.go_async()
print("Drone initialized, async executor running.")
print("Waiting 3 seconds for topics to connect...")
time.sleep(3)

## 2. Forward Camera Stream (10 seconds)

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = drone.camera.get_color_image_async()
    if color_image is not None:
        frame_count += 1
        img_widget.value = to_jpeg_bytes(color_image)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = f'Forward camera | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
label.value = f'Forward camera complete: {frame_count} frames in {elapsed:.1f}s = {frame_count/elapsed:.1f} FPS'

## 3. Depth Camera Stream (10 seconds)

In [ ]:
DURATION = 10
MAX_DEPTH_CM = 300  # 3 meters — practical indoor range

img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    depth_image = drone.camera.get_depth_image_async()
    if depth_image is not None:
        frame_count += 1
        # Scale to 0-255: close=255 (red), far=0 (blue) by inverting
        depth_normalized = np.clip(depth_image / MAX_DEPTH_CM, 0.0, 1.0)
        depth_display = (255 - (depth_normalized * 255)).astype(np.uint8)
        # Zero depth (no reading) → black
        depth_display[depth_image == 0] = 0
        depth_colored = cv.applyColorMap(depth_display, cv.COLORMAP_JET)
        depth_colored[depth_image == 0] = [0, 0, 0]
        img_widget.value = to_jpeg_bytes(depth_colored)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        center_dist = depth_image[depth_image.shape[0]//2, depth_image.shape[1]//2]
        label.value = f'Depth | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS | Center: {center_dist:.0f} cm | Range: 0-{MAX_DEPTH_CM} cm'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
label.value = f'Depth camera complete: {frame_count} frames in {elapsed:.1f}s = {frame_count/elapsed:.1f} FPS'

## 4. Nadir Camera Stream (10 seconds)

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    nadir_image = drone.camera.get_downward_image_async()
    if nadir_image is not None:
        frame_count += 1
        img_widget.value = to_jpeg_bytes(nadir_image)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = f'Nadir camera | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
label.value = f'Nadir camera complete: {frame_count} frames in {elapsed:.1f}s = {frame_count/elapsed:.1f} FPS'

## 5. IMU / Physics Data

In [ ]:
for i in range(5):
    accel = drone.physics.get_linear_acceleration()
    vel = drone.physics.get_linear_velocity()
    gyro = drone.physics.get_angular_velocity()
    alt = drone.physics.get_altitude()
    att = drone.physics.get_attitude()
    gps = drone.physics.get_gps()

    print(f"--- Sample {i+1} ---")
    print(f"  Accel:    ({accel[0]:7.2f}, {accel[1]:7.2f}, {accel[2]:7.2f}) m/s^2")
    print(f"  Velocity: ({vel[0]:7.2f}, {vel[1]:7.2f}, {vel[2]:7.2f}) m/s")
    print(f"  Gyro:     ({gyro[0]:7.3f}, {gyro[1]:7.3f}, {gyro[2]:7.3f}) rad/s")
    print(f"  Altitude: {alt} m")
    print(f"  Attitude: (P:{att[0]:6.1f}, R:{att[1]:6.1f}, Y:{att[2]:6.1f}) deg")
    print(f"  GPS:      ({gps[0]:.6f}, {gps[1]:.6f}, {gps[2]:.1f})")
    time.sleep(0.2)

accel_mag = np.linalg.norm(accel)
print(f"\nAccel magnitude: {accel_mag:.2f} m/s^2 (expected ~9.81 when stationary)")
if 8.0 < accel_mag < 12.0:
    print("PASS: IMU acceleration looks correct")
else:
    print("WARN: Acceleration magnitude outside expected range")

## 6. Coral EdgeTPU Inference

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=640, height=480)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
total_detections = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = drone.camera.get_color_image_async()
    detections = drone.detector.get_detections_async()
    if color_image is not None:
        frame_count += 1
        annotated = color_image.copy()
        for det in detections:
            cx, cy, w, h = det.bbox
            x1, y1 = int(cx - w / 2), int(cy - h / 2)
            x2, y2 = int(cx + w / 2), int(cy + h / 2)
            cv.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv.putText(annotated, f"{det.class_id} {det.score:.0%}", (x1, y1 - 8),
                       cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        total_detections += len(detections)
        img_widget.value = to_jpeg_bytes(annotated, resize=None)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = f'EdgeTPU | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS | Detections: {len(detections)} | Total: {total_detections}'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
label.value = f'EdgeTPU complete: {frame_count} frames in {elapsed:.1f}s = {frame_count/elapsed:.1f} FPS | Total detections: {total_detections}'

## 7. Xbox Controller (10 seconds)

Move the joysticks, press buttons, and pull triggers to verify input.

In [ ]:
DURATION = 10

left_label = widgets.Label(value='Left joy:  (0.00, 0.00)')
right_label = widgets.Label(value='Right joy: (0.00, 0.00)')
trigger_label = widgets.Label(value='Triggers:  L=0.00  R=0.00')
button_label = widgets.Label(value='Buttons:   none')
timer_label = widgets.Label(value='Starting...')

display(widgets.VBox([timer_label, left_label, right_label, trigger_label, button_label]))

t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Xbox Controller | {elapsed:.1f}s / {DURATION}s'

    lx, ly = drone.controller.get_joystick(drone.controller.Joystick.LEFT)
    rx, ry = drone.controller.get_joystick(drone.controller.Joystick.RIGHT)
    lt = drone.controller.get_trigger(drone.controller.Trigger.LEFT)
    rt = drone.controller.get_trigger(drone.controller.Trigger.RIGHT)

    left_label.value = f'Left joy:  ({lx:+.2f}, {ly:+.2f})'
    right_label.value = f'Right joy: ({rx:+.2f}, {ry:+.2f})'
    trigger_label.value = f'Triggers:  L={lt:.2f}  R={rt:.2f}'

    pressed = [b.name for b in drone.controller.Button if drone.controller.is_down(b)]
    button_label.value = f'Buttons:   {", ".join(pressed) if pressed else "none"}'

    time.sleep(0.05)

timer_label.value = f'Xbox Controller test complete ({DURATION}s)'

## 8. RC Transmitter (10 seconds)

Move the RC sticks and flip switches. Reads directly from `/mavros/rc/in`.

In [ ]:
import rclpy as ros2
from mavros_msgs.msg import RCIn
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSDurabilityPolicy

# Create a temporary subscriber for /mavros/rc/in
rc_node = ros2.create_node("rc_test_sub")
qos = QoSProfile(depth=1)
qos.reliability = QoSReliabilityPolicy.BEST_EFFORT
qos.durability = QoSDurabilityPolicy.VOLATILE

rc_channels = [0] * 8
def rc_cb(msg):
    global rc_channels
    rc_channels = list(msg.channels)

rc_sub = rc_node.create_subscription(RCIn, "/mavros/rc/in", rc_cb, qos)
drone._DroneReal__executor.add_node(rc_node)

DURATION = 10
CH_NAMES = ["Roll/Ail", "Pitch/Ele", "Throttle", "Yaw/Rud", "SwA (Arm)", "SwB (Mode)", "SwC (Loit)", "SwD (Offb)"]

timer_label = widgets.Label(value='Starting...')
ch_labels = [widgets.Label(value=f'  CH{i} {CH_NAMES[i] if i < len(CH_NAMES) else "":12s}: ----') for i in range(8)]
display(widgets.VBox([timer_label] + ch_labels))

t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'RC Transmitter | {elapsed:.1f}s / {DURATION}s | RSSI: active'
    for i in range(min(8, len(rc_channels))):
        pwm = rc_channels[i]
        # Visualize as a bar: 1000=left, 1500=center, 2000=right
        normalized = (pwm - 1000) / 1000.0  # 0.0 to 1.0
        bar_len = int(normalized * 20)
        bar = '█' * bar_len + '░' * (20 - bar_len)
        name = CH_NAMES[i] if i < len(CH_NAMES) else f"CH{i}"
        ch_labels[i].value = f'  CH{i} {name:12s}: {pwm:4d}  [{bar}]'
    time.sleep(0.05)

timer_label.value = f'RC Transmitter test complete ({DURATION}s)'
drone._DroneReal__executor.remove_node(rc_node)
rc_node.destroy_node()

## 9. Vehicle State

Reads flight mode, armed status, and landed state from the Pixhawk.

In [ ]:
DURATION = 10

connected_label = widgets.Label(value='Connected: ...')
armed_label = widgets.Label(value='Armed: ...')
mode_label = widgets.Label(value='Mode: ...')
landed_label = widgets.Label(value='Landed state: ...')
offboard_label = widgets.Label(value='Offboard: ...')
timer_label = widgets.Label(value='Starting...')

display(widgets.VBox([timer_label, connected_label, armed_label, mode_label, landed_label, offboard_label]))

t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Vehicle State | {elapsed:.1f}s / {DURATION}s'

    connected_label.value = f'Connected:    {"YES" if drone.state.is_connected() else "NO"}'
    armed_label.value = f'Armed:        {"YES — MOTORS LIVE" if drone.state.is_armed() else "No"}'
    mode_label.value = f'Flight mode:  {drone.state.get_mode() or "(unknown)"}'
    landed_label.value = f'Landed state: {drone.state.get_landed_state().name}'
    offboard_label.value = f'Offboard:     {"YES — Pi has control" if drone.state.is_offboard() else "No"}'

    time.sleep(0.1)

timer_label.value = f'Vehicle State test complete ({DURATION}s)'

## 10. Mux Test (10 seconds)

Shows the mux output and current mode. Test by holding bumpers:
- **No bumper** → IDLE (zero velocity)
- **LB held** → MANUAL (Xbox sticks forwarded)
- **RB held** → AUTO: runs a simulated flight sequence (takeoff → fly forward+right 3s → land). Releasing RB cancels immediately.

In [ ]:
import rclpy as ros2
from geometry_msgs.msg import TwistStamped as _TwistStamped

# Subscribe to mux output directly
mux_node = ros2.create_node("mux_test_sub")
mux_output = [_TwistStamped()]
def _mux_cb(msg):
    mux_output[0] = msg
mux_node.create_subscription(_TwistStamped, '/mavros/setpoint_velocity/cmd_vel', _mux_cb, 10)
drone._DroneReal__executor.add_node(mux_node)

DURATION = 30

mode_label = widgets.Label(value='Mux mode: ...')
output_label = widgets.Label(value='Output: ...')
lb_rb_label = widgets.Label(value='Bumpers: ...')
phase_label = widgets.Label(value='Auto phase: ---')
timer_label = widgets.Label(value='Starting...')

display(widgets.VBox([timer_label, lb_rb_label, mode_label, phase_label, output_label]))

# Auto flight sequence state
auto_phase = "IDLE"
phase_start = None

t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Mux Test | {elapsed:.1f}s / {DURATION}s'

    lb = drone.controller.is_down(drone.controller.Button.LB)
    rb = drone.controller.is_down(drone.controller.Button.RB)

    # Determine mode
    if lb and not rb:
        mode = 'MANUAL (LB) — Xbox sticks active'
        auto_phase = "IDLE"
        phase_start = None
        drone.flight.stop()
    elif rb and not lb:
        mode = 'AUTO (RB) — flight sequence active'

        # Run the auto flight sequence
        if auto_phase == "IDLE":
            auto_phase = "TAKEOFF"
            phase_start = time.monotonic()
            drone.flight.takeoff()

        phase_elapsed = time.monotonic() - phase_start

        if auto_phase == "TAKEOFF" and phase_elapsed > 3.0:
            auto_phase = "FLY"
            phase_start = time.monotonic()
            drone.flight.send_pcmd(0.5, 0.3, 0.0, 0.0)  # forward + right

        elif auto_phase == "FLY" and phase_elapsed > 3.0:
            auto_phase = "LAND"
            phase_start = time.monotonic()
            drone.flight.land()

        elif auto_phase == "LAND" and phase_elapsed > 3.0:
            auto_phase = "DONE"
            drone.flight.stop()

    else:
        mode = 'IDLE — zero velocity'
        if auto_phase not in ("IDLE", "DONE"):
            auto_phase = "CANCELLED"
            drone.flight.stop()
        elif auto_phase == "CANCELLED":
            auto_phase = "IDLE"
        phase_start = None

    lb_rb_label.value = f'Bumpers:    LB={"HELD" if lb else "---"}  RB={"HELD" if rb else "---"}'
    mode_label.value = f'Mux mode:   {mode}'
    phase_label.value = f'Auto phase: {auto_phase}' + (f' ({time.monotonic() - phase_start:.1f}s)' if phase_start else '')

    m = mux_output[0]
    output_label.value = (
        f'Output:     pitch={m.twist.linear.x:+.3f}  roll={m.twist.linear.y:+.3f}  '
        f'throt={m.twist.linear.z:+.3f}  yaw={m.twist.angular.z:+.3f}'
    )
    time.sleep(0.05)

drone.flight.stop()
timer_label.value = f'Mux test complete ({DURATION}s)'
drone._DroneReal__executor.remove_node(mux_node)
mux_node.destroy_node()

## 11. Diagnostics

In [ ]:
diag = drone.telemetry.get_diagnostics()

if diag:
    for name, info in diag.items():
        level = info["level"]
        if isinstance(level, (bytes, bytearray)):
            level = int.from_bytes(level, byteorder='big')
        level_str = {0: "OK", 1: "WARN", 2: "ERROR", 3: "STALE"}.get(level, f"LVL:{level}")
        print(f"[{level_str}] {name}: {info['message']}")
        for k, v in info.items():
            if k not in ("level", "message", "hardware_id"):
                print(f"    {k}: {v}")
        print()
else:
    print("No diagnostics received yet.")

## 12. Summary

In [ ]:
results = {
    "Forward camera (/camera/forward)": drone.camera.get_color_image_async() is not None,
    "Depth camera (/camera/depth)": drone.camera.get_depth_image_async() is not None,
    "Nadir camera (/camera/nadir)": drone.camera.get_downward_image_async() is not None,
    "IMU (/mavros/imu/data)": np.any(drone.physics.get_linear_acceleration() != 0),
    "Velocity (/velocity)": True,
    "Xbox controller (/joy)": drone.controller.get_joystick(drone.controller.Joystick.LEFT) is not None,
    "Vehicle state (/mavros/state)": drone.state.is_connected(),
    "Mux (/mavros/setpoint_velocity)": True,  # verified interactively above
    "EdgeTPU (/edgetpu/inference)": len(drone.detector.get_detections_async()) >= 0,
    "Diagnostics (/diagnostics)": len(drone.telemetry.get_diagnostics()) > 0,
}

gps = drone.physics.get_gps()
gps_ok = np.any(gps != 0)

print("=" * 60)
print("  UAV Neo Async Core Test — Results")
print("=" * 60)
all_pass = True
for name, ok in results.items():
    status = "PASS" if ok else "FAIL"
    if not ok:
        all_pass = False
    print(f"  [{status}] {name}")
print(f"  [{'PASS' if gps_ok else 'SKIP'}] GPS (/nav) {'— no GPS module installed' if not gps_ok else ''}")
print("=" * 60)
print(f"  Overall: {'ALL PASS' if all_pass else 'SOME FAILURES — check above'}")
print("=" * 60)